In [3]:
import json, os, time, warnings, logging
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_recall_curve, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve)
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.model_selection import train_test_split

In [4]:
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
sns.set_style("whitegrid")

In [5]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

BASE_DIR = Path.cwd()
PATHS = {
    "processed": BASE_DIR / "dataset_processed.csv",
    "splits": BASE_DIR / "split_indices.npz",
    "models_dir": BASE_DIR / "models",
    "scaler": BASE_DIR / "scaler.pkl",
    "plots_dir": BASE_DIR / "plots",
}
PATHS["plots_dir"].mkdir(exist_ok=True)

TARGET_FPR = 1e-3
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("Device count:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
print(f"Device: {DEVICE}")

CUDA available: True
CUDA version: 12.6
Device count: 1
GPU name: NVIDIA GeForce RTX 2060
Device: cuda


In [6]:
full_df = pd.read_csv(PATHS["processed"])
splits = np.load(PATHS["splits"], allow_pickle=True)

train_idx = splits["train_idx"]
val_idx = splits["val_idx"]
test_id_idx = splits["test_id_idx"]
test_ood_idx = splits["test_ood_idx"]
feature_cols_final = [c for c in full_df.columns if c.startswith("nlp_") or c.startswith("isnan_")]

# Domains and labels
X_domain = full_df["domain"].values
y_all = full_df["label"].values

# Load saved models
trained_models = {}
for name in ["LightGBM", "XGBoost", "MLP"]:
    model_path = PATHS["models_dir"] / f"{name}.pkl"
    if model_path.exists():
        trained_models[name] = joblib.load(model_path)
        print(f"Loaded {name}")

# Load scaler for MLP
scaler = joblib.load(PATHS["scaler"])

# Load saved feature matrix for sklearn models (or reconstruct from full_df)
X_features = full_df[feature_cols_final].astype("float32")

Loaded LightGBM
Loaded XGBoost
Loaded MLP


In [9]:
MAX_LEN = 75
VOCAB = {chr(i): i + 1 for i in range(128)}
VOCAB["<PAD>"] = 0
VOCAB["<UNK>"] = 129

def encode_domain(domain, max_len=MAX_LEN):
    seq = [VOCAB.get(c, VOCAB["<UNK>"]) for c in domain.lower()[:max_len]]
    seq += [VOCAB["<PAD>"]] * (max_len - len(seq))
    return np.array(seq)

print("Encoding domains ONCE...")
X_train_enc    = np.array([encode_domain(d) for d in X_domain[train_idx]],    dtype=np.int64)
X_val_enc      = np.array([encode_domain(d) for d in X_domain[val_idx]],      dtype=np.int64)
X_test_id_enc  = np.array([encode_domain(d) for d in X_domain[test_id_idx]],  dtype=np.int64)
X_test_ood_enc = np.array([encode_domain(d) for d in X_domain[test_ood_idx]], dtype=np.int64)

X_train_enc    = torch.from_numpy(X_train_enc)
X_val_enc      = torch.from_numpy(X_val_enc)
X_test_id_enc  = torch.from_numpy(X_test_id_enc)
X_test_ood_enc = torch.from_numpy(X_test_ood_enc)

y_train_t    = torch.from_numpy(y_all[train_idx]).float()
y_val_t      = torch.from_numpy(y_all[val_idx]).float()

class PreEncodedDataset(Dataset):
    def __init__(self, X_enc, y):
        self.X, self.y = X_enc, y
    def __len__(self): return len(self.X)
    def __getitem__(self, idx): return self.X[idx], self.y[idx]

print(f"Train shape: {X_train_enc.shape}")

Encoding domains ONCE...
Train shape: torch.Size([1281254, 75])


In [10]:
class CharCNN(nn.Module):
    def __init__(self, vocab_size=130, embed_dim=32, num_filters=64, kernel_size=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv1 = nn.Conv1d(embed_dim, num_filters, kernel_size)
        self.pool = nn.AdaptiveMaxPool1d(1)
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(num_filters, 1)
    def forward(self, x):
        x = self.embedding(x).permute(0, 2, 1)
        x = torch.relu(self.conv1(x))
        x = self.pool(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x).squeeze(-1)

def train_char_cnn(X_enc, y_t, X_val_enc, y_val_t, epochs=5, batch_size=2048, lr=1e-3):
    train_loader = DataLoader(
        PreEncodedDataset(X_enc, y_t),
        batch_size=batch_size, shuffle=True,
        num_workers=0,             # <-- КРИТИЧНО: 0, а не 2!
        pin_memory=True,
    )
    val_loader = DataLoader(
        PreEncodedDataset(X_val_enc, y_val_t),
        batch_size=batch_size, shuffle=False,
        num_workers=0, pin_memory=True,
    )
    model = CharCNN().to(DEVICE)
    print(f"Model is on: {next(model.parameters()).device}")   # Проверка!

    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.BCEWithLogitsLoss()
    best_f1, best_state = 0, None

    for epoch in range(epochs):
        model.train()
        t0 = time.perf_counter()
        for x, y in train_loader:
            x = x.to(DEVICE, non_blocking=True)
            y = y.to(DEVICE, non_blocking=True)
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
        t_train = time.perf_counter() - t0

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(DEVICE, non_blocking=True)
                preds.extend(torch.sigmoid(model(x)).cpu().numpy())
                labels.extend(y.numpy())
        f1 = f1_score(labels, (np.array(preds) >= 0.5).astype(int))
        print(f"Epoch {epoch+1}/{epochs} | train {t_train:.1f}s | Val F1: {f1:.4f}")
        if f1 > best_f1:
            best_f1, best_state = f1, model.state_dict().copy()
    model.load_state_dict(best_state)
    return model

char_cnn = train_char_cnn(X_train_enc, y_train_t, X_val_enc, y_val_t, epochs=5, batch_size=2048)

Model is on: cuda:0
Epoch 1/5 | train 17.8s | Val F1: 0.8840
Epoch 2/5 | train 10.0s | Val F1: 0.9009
Epoch 3/5 | train 10.0s | Val F1: 0.9089
Epoch 4/5 | train 12.6s | Val F1: 0.9139
Epoch 5/5 | train 10.5s | Val F1: 0.9154


In [11]:
def predict_char_cnn(model, X_enc, batch_size=2048):
    loader = DataLoader(PreEncodedDataset(X_enc, torch.zeros(len(X_enc))),
                        batch_size=batch_size, shuffle=False, num_workers=0)
    model.eval()
    probs = []
    with torch.no_grad():
        for x, _ in loader:
            x = x.to(DEVICE, non_blocking=True)
            probs.extend(torch.sigmoid(model(x)).cpu().numpy())
    return np.array(probs)

p_id_char  = predict_char_cnn(char_cnn, X_test_id_enc)
p_ood_char = predict_char_cnn(char_cnn, X_test_ood_enc)

# Metrics
def compute_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (y_prob >= threshold).astype(int)
    return {
        "f1": f1_score(y_true, y_pred),
        "pr_auc": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob),
    }

metrics_char_id = compute_metrics(y_all[test_id_idx], p_id_char)
metrics_char_ood = compute_metrics(y_all[test_ood_idx], p_ood_char)
print("Char-CNN test_id:", metrics_char_id)
print("Char-CNN test_ood:", metrics_char_ood)

Char-CNN test_id: {'f1': 0.9145225095628862, 'pr_auc': 0.9772382164860309, 'roc_auc': 0.9786497182017}
Char-CNN test_ood: {'f1': 0.8960314429677095, 'pr_auc': 0.9715577765388618, 'roc_auc': 0.9642918827121103}


In [12]:
# Use a small pre-trained model for speed; replace with DomURLs_BERT if available
MODEL_NAME = "distilbert-base-uncased"  # lightweight; use "bert-base-uncased" for better quality

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, 
    use_fast=False
)
model_bert = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

# Prepare datasets
class BertDGA(Dataset):
    def __init__(self, domains, labels, tokenizer, max_len=64):
        self.domains = domains
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    def __len__(self):
        return len(self.domains)
    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.domains[idx], truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_ds_bert = BertDGA(X_domain[train_idx], y_all[train_idx], tokenizer)
val_ds_bert = BertDGA(X_domain[val_idx], y_all[val_idx], tokenizer)

training_args = TrainingArguments(
    output_dir="./bert_dga",
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)

def compute_metrics_bert(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"f1": f1_score(labels, preds)}

trainer = Trainer(
    model=model_bert,
    args=training_args,
    train_dataset=train_ds_bert,
    eval_dataset=val_ds_bert,
    compute_metrics=compute_metrics_bert,
)

print("Training BERT...")
trainer.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1823.78it/s]


Training BERT...
{'loss': '0.4473', 'grad_norm': '5.871', 'learning_rate': '4.95e-05', 'epoch': '0.002498'}
{'loss': '0.2384', 'grad_norm': '8.086', 'learning_rate': '4.996e-05', 'epoch': '0.004995'}
{'loss': '0.2145', 'grad_norm': '6.177', 'learning_rate': '4.992e-05', 'epoch': '0.007493'}
{'loss': '0.1865', 'grad_norm': '4.181', 'learning_rate': '4.988e-05', 'epoch': '0.00999'}
{'loss': '0.185', 'grad_norm': '2.218', 'learning_rate': '4.983e-05', 'epoch': '0.01249'}
{'loss': '0.1716', 'grad_norm': '4.744', 'learning_rate': '4.979e-05', 'epoch': '0.01499'}
{'loss': '0.1402', 'grad_norm': '2.823', 'learning_rate': '4.975e-05', 'epoch': '0.01748'}
{'loss': '0.1375', 'grad_norm': '4.314', 'learning_rate': '4.971e-05', 'epoch': '0.01998'}
{'loss': '0.137', 'grad_norm': '0.2963', 'learning_rate': '4.967e-05', 'epoch': '0.02248'}
{'loss': '0.142', 'grad_norm': '3.32', 'learning_rate': '4.963e-05', 'epoch': '0.02498'}
{'loss': '0.1207', 'grad_norm': '11.69', 'learning_rate': '4.958e-05', 'ep

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.57it/s]


{'loss': '0.04263', 'grad_norm': '11.61', 'learning_rate': '3.334e-05', 'epoch': '1.001'}
{'loss': '0.04497', 'grad_norm': '0.05938', 'learning_rate': '3.329e-05', 'epoch': '1.004'}
{'loss': '0.04634', 'grad_norm': '3.732', 'learning_rate': '3.325e-05', 'epoch': '1.006'}
{'loss': '0.04369', 'grad_norm': '0.196', 'learning_rate': '3.321e-05', 'epoch': '1.009'}
{'loss': '0.03684', 'grad_norm': '0.03097', 'learning_rate': '3.317e-05', 'epoch': '1.011'}
{'loss': '0.0361', 'grad_norm': '31.15', 'learning_rate': '3.313e-05', 'epoch': '1.014'}
{'loss': '0.04308', 'grad_norm': '0.2875', 'learning_rate': '3.309e-05', 'epoch': '1.016'}
{'loss': '0.04936', 'grad_norm': '0.3576', 'learning_rate': '3.304e-05', 'epoch': '1.019'}
{'loss': '0.0477', 'grad_norm': '0.07709', 'learning_rate': '3.3e-05', 'epoch': '1.021'}
{'loss': '0.04671', 'grad_norm': '0.0657', 'learning_rate': '3.296e-05', 'epoch': '1.024'}
{'loss': '0.05206', 'grad_norm': '1.968', 'learning_rate': '3.292e-05', 'epoch': '1.026'}
{'los

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.61it/s]


{'loss': '0.02997', 'grad_norm': '0.02714', 'learning_rate': '1.667e-05', 'epoch': '2'}
{'loss': '0.02741', 'grad_norm': '0.008027', 'learning_rate': '1.663e-05', 'epoch': '2.003'}
{'loss': '0.02589', 'grad_norm': '0.1425', 'learning_rate': '1.659e-05', 'epoch': '2.005'}
{'loss': '0.02903', 'grad_norm': '0.2587', 'learning_rate': '1.655e-05', 'epoch': '2.008'}
{'loss': '0.02138', 'grad_norm': '0.1013', 'learning_rate': '1.651e-05', 'epoch': '2.01'}
{'loss': '0.02433', 'grad_norm': '0.4181', 'learning_rate': '1.646e-05', 'epoch': '2.013'}
{'loss': '0.01752', 'grad_norm': '0.2737', 'learning_rate': '1.642e-05', 'epoch': '2.015'}
{'loss': '0.0231', 'grad_norm': '0.04935', 'learning_rate': '1.638e-05', 'epoch': '2.018'}
{'loss': '0.02516', 'grad_norm': '5.145', 'learning_rate': '1.634e-05', 'epoch': '2.02'}
{'loss': '0.02581', 'grad_norm': '37.73', 'learning_rate': '1.63e-05', 'epoch': '2.023'}
{'loss': '0.02947', 'grad_norm': '2.347', 'learning_rate': '1.626e-05', 'epoch': '2.025'}
{'loss

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.91it/s]


{'train_runtime': '1.675e+04', 'train_samples_per_second': '229.4', 'train_steps_per_second': '7.17', 'train_loss': '0.04658', 'epoch': '3'}


TrainOutput(global_step=120120, training_loss=0.0465848294928139, metrics={'train_runtime': 16754.3051, 'train_samples_per_second': 229.419, 'train_steps_per_second': 7.17, 'train_loss': 0.0465848294928139, 'epoch': 3.0})

In [15]:
def predict_bert(model, domains, batch_size=64):
    model.eval()
    all_probs = []
    domains = list(domains) if not isinstance(domains, list) else domains
    for i in range(0, len(domains), batch_size):
        batch_domains = domains[i:i+batch_size]
        enc = tokenizer(batch_domains, truncation=True, padding=True, max_length=64, return_tensors="pt")
        enc = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            logits = model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, 1].cpu().numpy()
            all_probs.extend(probs)
    return np.array(all_probs)

p_id_bert = predict_bert(model_bert, X_domain[test_id_idx])
p_ood_bert = predict_bert(model_bert, X_domain[test_ood_idx])

metrics_bert_id = compute_metrics(y_all[test_id_idx], p_id_bert)
metrics_bert_ood = compute_metrics(y_all[test_ood_idx], p_ood_bert)
print("BERT test_id:", metrics_bert_id)
print("BERT test_ood:", metrics_bert_ood)

BERT test_id: {'f1': 0.9862730757793726, 'pr_auc': 0.9988593085305133, 'roc_auc': 0.9990287409227255}
BERT test_ood: {'f1': 0.9293688902791898, 'pr_auc': 0.9878988158023206, 'roc_auc': 0.9832086986170516}
